# Intraday Trading Strategies — Conversational Multi-Agent System with AutoGen

This notebook builds a **production-grade multi-agent trading system** using Microsoft AutoGen. Specialized agents collaborate in conversation to analyze intraday market data, generate signals, manage risk, and report performance.

---

## Architecture

```
User Query
     │
     ▼
┌─────────────────────────────────────────────────────────────────┐
│                  Trading Floor GroupChat                        │
│                                                                 │
│  MarketDataAgent ──► TechnicalAnalyst ──► StrategyAgent        │
│         │                                       │               │
│         └───────────► RiskManager ◄─────────────┘               │
│                            │                                    │
│                      TradeExecutor                              │
│                            │                                    │
│                   PerformanceAnalyst                            │
└─────────────────────────────────────────────────────────────────┘
```

## Intraday Strategies Covered

| Strategy | Description | Timeframe |
|---|---|---|
| **Opening Range Breakout (ORB)** | Trade breakout from the first 30-min high/low | 30-min |
| **VWAP Reversion** | Buy below VWAP, sell at VWAP | 5-min |
| **RSI + MACD Momentum** | Enter on RSI oversold + MACD bullish crossover | 15-min |
| **Bollinger Band Squeeze** | Trade breakout after low-volatility consolidation | 5-min |

## Table of Contents

1. Installation and Setup
2. Market Data Simulation Tools
3. Technical Analysis Tools
4. Strategy Signal Tools
5. Risk Management Tools
6. Performance Analytics Tools
7. Agent Definitions
8. **Implementation 1** — Two-Agent Technical Analysis Debate
9. **Implementation 2** — GroupChat Trading Floor (6 Agents)
10. **Implementation 3** — Sequential Strategy Pipeline
11. **Implementation 4** — Interactive Conversational Trading Assistant
12. Best Practices for Trading Multi-Agent Systems

---
## 1. Installation and Setup

In [ ]:
%pip install pyautogen autogen-agentchat autogen-ext[openai] --upgrade --quiet
%pip install pandas numpy scipy matplotlib seaborn python-dotenv --quiet

In [3]:
import os
import json
import math
import random
import asyncio
from datetime import datetime, timedelta
from typing import List, Dict, Any, Optional

import numpy as np
import pandas as pd
import matplotlib
#matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
# import seaborn as sns

from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat, SelectorGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_ext.models.openai import OpenAIChatCompletionClient

os.makedirs('./trading_workspace', exist_ok=True)
print('Imports successful. Trading workspace ready.')

Imports successful. Trading workspace ready.


In [ ]:
# ============================================================
# API KEY CONFIGURATION
# ============================================================
api_key = os.environ.get('OPENAI_API_KEY', '')
assert api_key, 'API key must be set'

model_client = OpenAIChatCompletionClient(
    model='gpt-4o-mini',
    api_key=api_key,
)

print(f'Model client ready: gpt-4o-mini')

---
## 2. Market Data Simulation Tools

These functions simulate realistic intraday OHLCV data using geometric Brownian motion with intraday volatility clustering (higher volatility at open and close, lower mid-day).

In [5]:
# ============================================================
# MARKET DATA SIMULATION ENGINE
# Uses Geometric Brownian Motion + intraday volatility pattern
# ============================================================

def generate_intraday_data(ticker: str, date: str = None,
                            interval_minutes: int = 5,
                            seed: int = 42) -> dict:
    """
    Generate realistic intraday OHLCV bars for a ticker.
    Returns a dict with 'ticker', 'date', 'interval', 'bars' (list of OHLCV dicts).
    """
    random.seed(seed)
    np.random.seed(seed)

    if date is None:
        date = datetime.now().strftime('%Y-%m-%d')

    # Base price per ticker
    base_prices = {
        'RELIANCE': 2850.0, 'TCS': 3920.0, 'HDFC': 1680.0,
        'INFY': 1540.0,     'ICICI': 1120.0, 'WIPRO': 480.0,
        'AAPL': 189.0,      'MSFT': 415.0,   'NVDA': 875.0,
        'TSLA': 248.0,      'SPY': 520.0,    'QQQ': 445.0,
    }
    S0 = base_prices.get(ticker.upper(), 1000.0)

    # Session: 9:15 AM to 3:30 PM IST (375 minutes) or 9:30–4:00 ET
    market_open = datetime.strptime(f'{date} 09:15:00', '%Y-%m-%d %H:%M:%S')
    total_minutes = 375
    n_bars = total_minutes // interval_minutes

    # Intraday volatility multiplier (U-shape: high at open/close)
    def vol_multiplier(bar_index, total_bars):
        x = bar_index / total_bars
        return 1.5 * (1 - 4 * (x - 0.5) ** 2) + 0.5

    annual_vol = 0.28
    dt = interval_minutes / (252 * 375)
    mu = 0.0

    bars = []
    S = S0
    volume_base = random.randint(50000, 200000)

    for i in range(n_bars):
        vol = annual_vol * vol_multiplier(i, n_bars)
        ts = market_open + timedelta(minutes=i * interval_minutes)

        # Generate open from previous close
        if i == 0:
            # Gap-up or gap-down at open
            gap = np.random.normal(0, 0.003)
            open_price = round(S * (1 + gap), 2)
        else:
            open_price = round(S, 2)

        # Intrabar GBM
        n_ticks = 10
        prices = [open_price]
        for _ in range(n_ticks - 1):
            dS = prices[-1] * (mu * dt/n_ticks + vol * math.sqrt(dt/n_ticks) * np.random.randn())
            prices.append(max(prices[-1] + dS, 0.01))

        high_price = round(max(prices) * (1 + abs(np.random.normal(0, 0.0005))), 2)
        low_price  = round(min(prices) * (1 - abs(np.random.normal(0, 0.0005))), 2)
        close_price = round(prices[-1], 2)

        # Volume is higher at open/close
        vol_mult = vol_multiplier(i, n_bars)
        volume = int(volume_base * vol_mult * (0.8 + 0.4 * random.random()))

        bars.append({
            'timestamp': ts.strftime('%H:%M'),
            'open': open_price,
            'high': high_price,
            'low': low_price,
            'close': close_price,
            'volume': volume
        })
        S = close_price

    return {
        'ticker': ticker.upper(),
        'date': date,
        'interval_minutes': interval_minutes,
        'total_bars': len(bars),
        'open_price': bars[0]['open'],
        'current_price': bars[-1]['close'],
        'day_high': round(max(b['high'] for b in bars), 2),
        'day_low': round(min(b['low'] for b in bars), 2),
        'total_volume': sum(b['volume'] for b in bars),
        'bars': bars[:20]  # Return first 20 bars for agent context
    }


def get_market_snapshot(tickers: list) -> dict:
    """Get current-day snapshot for multiple tickers."""
    snapshots = {}
    for i, ticker in enumerate(tickers):
        data = generate_intraday_data(ticker, seed=42 + i)
        bars = data['bars']
        mid = len(bars) // 2
        current = bars[mid]
        prev_close = data['open_price'] * (1 + np.random.normal(-0.001, 0.005))
        change_pct = round((current['close'] - prev_close) / prev_close * 100, 2)
        snapshots[ticker.upper()] = {
            'price': current['close'],
            'change_pct': change_pct,
            'volume': current['volume'],
            'day_high': data['day_high'],
            'day_low': data['day_low'],
            'trend': 'BULLISH' if change_pct > 0.3 else 'BEARISH' if change_pct < -0.3 else 'NEUTRAL'
        }
    return {'timestamp': datetime.now().strftime('%H:%M:%S'), 'quotes': snapshots}


# Quick test
snap = get_market_snapshot(['RELIANCE', 'TCS', 'INFY'])
print('Market Snapshot:')
for ticker, q in snap['quotes'].items():
    print(f'  {ticker}: {q["price"]:>10.2f}  {q["change_pct"]:+.2f}%  {q["trend"]}')

Market Snapshot:
  RELIANCE:    2847.00  -0.94%  BEARISH
  TCS:    3903.77  -0.12%  NEUTRAL
  INFY:    1533.55  -0.32%  BEARISH


---
## 3. Technical Analysis Tools

Pure Python implementations of key intraday indicators — no TA-Lib dependency required.

In [6]:
# ============================================================
# TECHNICAL ANALYSIS ENGINE
# All indicators implemented from scratch (no TA-Lib needed)
# ============================================================

def _bars_to_df(ticker: str, seed: int = 42) -> pd.DataFrame:
    """Helper: generate full intraday data as a DataFrame."""
    raw = generate_intraday_data(ticker, interval_minutes=5, seed=seed)
    df = pd.DataFrame(raw['bars'])
    return df


def compute_vwap(ticker: str) -> dict:
    """Compute intraday VWAP and current price distance from VWAP."""
    df = _bars_to_df(ticker)
    typical_price = (df['high'] + df['low'] + df['close']) / 3
    cumulative_tp_vol = (typical_price * df['volume']).cumsum()
    cumulative_vol = df['volume'].cumsum()
    df['vwap'] = cumulative_tp_vol / cumulative_vol

    current_close = df['close'].iloc[-1]
    current_vwap = round(df['vwap'].iloc[-1], 2)
    deviation_pct = round((current_close - current_vwap) / current_vwap * 100, 3)

    return {
        'ticker': ticker.upper(),
        'current_price': round(current_close, 2),
        'vwap': current_vwap,
        'deviation_pct': deviation_pct,
        'position_vs_vwap': 'ABOVE' if deviation_pct > 0 else 'BELOW',
        'signal': 'BUY' if deviation_pct < -0.5 else 'SELL' if deviation_pct > 0.5 else 'NEUTRAL',
        'interpretation': (
            f'Price is {abs(deviation_pct):.3f}% {"below" if deviation_pct < 0 else "above"} VWAP. '
            f'{"Mean-reversion BUY setup" if deviation_pct < -0.5 else "Mean-reversion SELL setup" if deviation_pct > 0.5 else "Price near VWAP, no clear bias"}'
        )
    }


def compute_rsi(ticker: str, period: int = 14) -> dict:
    """Compute RSI(14) on 5-min closes."""
    df = _bars_to_df(ticker)
    delta = df['close'].diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1/period, min_periods=period).mean()
    avg_loss = loss.ewm(alpha=1/period, min_periods=period).mean()
    rs = avg_gain / avg_loss.replace(0, 1e-10)
    rsi_series = 100 - (100 / (1 + rs))
    current_rsi = round(rsi_series.iloc[-1], 2)

    if current_rsi < 30:
        condition, signal = 'OVERSOLD', 'BUY'
    elif current_rsi > 70:
        condition, signal = 'OVERBOUGHT', 'SELL'
    else:
        condition, signal = 'NEUTRAL', 'HOLD'

    return {
        'ticker': ticker.upper(),
        'rsi': current_rsi,
        'period': period,
        'condition': condition,
        'signal': signal,
        'interpretation': f'RSI({period})={current_rsi:.1f} — {condition}. Momentum signal: {signal}'
    }


def compute_macd(ticker: str, fast: int = 12, slow: int = 26, signal: int = 9) -> dict:
    """Compute MACD line, signal line, and histogram."""
    df = _bars_to_df(ticker)
    ema_fast = df['close'].ewm(span=fast, adjust=False).mean()
    ema_slow = df['close'].ewm(span=slow, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    histogram = macd_line - signal_line

    current_macd = round(macd_line.iloc[-1], 4)
    current_signal = round(signal_line.iloc[-1], 4)
    current_hist = round(histogram.iloc[-1], 4)
    prev_hist = round(histogram.iloc[-2], 4)

    crossover = (
        'BULLISH_CROSSOVER' if current_macd > current_signal and prev_hist <= 0 and current_hist > 0
        else 'BEARISH_CROSSOVER' if current_macd < current_signal and prev_hist >= 0 and current_hist < 0
        else 'BULLISH' if current_macd > current_signal
        else 'BEARISH'
    )

    return {
        'ticker': ticker.upper(),
        'macd': current_macd,
        'signal_line': current_signal,
        'histogram': current_hist,
        'crossover': crossover,
        'signal': 'BUY' if 'BULLISH' in crossover else 'SELL',
        'interpretation': f'MACD={current_macd:.4f} | Signal={current_signal:.4f} | Histogram={current_hist:.4f} — {crossover}'
    }


def compute_bollinger_bands(ticker: str, period: int = 20, std_dev: float = 2.0) -> dict:
    """Compute Bollinger Bands and detect squeeze/breakout."""
    df = _bars_to_df(ticker)
    rolling_mean = df['close'].rolling(period).mean()
    rolling_std = df['close'].rolling(period).std()
    upper = rolling_mean + std_dev * rolling_std
    lower = rolling_mean - std_dev * rolling_std
    bandwidth = (upper - lower) / rolling_mean * 100

    current_close = df['close'].iloc[-1]
    current_upper = round(upper.iloc[-1], 2)
    current_lower = round(lower.iloc[-1], 2)
    current_mid = round(rolling_mean.iloc[-1], 2)
    current_bw = round(bandwidth.iloc[-1], 3)
    avg_bw = round(bandwidth.rolling(50).mean().iloc[-1], 3)

    squeeze = current_bw < avg_bw * 0.75
    if current_close > current_upper:
        position, signal = 'ABOVE_UPPER', 'SELL'
    elif current_close < current_lower:
        position, signal = 'BELOW_LOWER', 'BUY'
    else:
        position, signal = 'INSIDE_BANDS', 'HOLD'

    return {
        'ticker': ticker.upper(),
        'current_price': round(current_close, 2),
        'upper_band': current_upper,
        'middle_band': current_mid,
        'lower_band': current_lower,
        'bandwidth_pct': current_bw,
        'squeeze_detected': squeeze,
        'price_position': position,
        'signal': signal,
        'interpretation': (
            f'BB({period},{std_dev}): Upper={current_upper}, Mid={current_mid}, Lower={current_lower}. '
            f'Bandwidth={current_bw:.2f}%. '
            f'{"SQUEEZE DETECTED — imminent breakout likely!" if squeeze else "Normal volatility"}. '
            f'Price is {position}.'
        )
    }


def compute_opening_range(ticker: str, range_minutes: int = 30) -> dict:
    """Compute Opening Range Breakout levels."""
    df = _bars_to_df(ticker, seed=99)
    interval = 5
    n_bars = range_minutes // interval
    or_bars = df.iloc[:n_bars]
    or_high = round(or_bars['high'].max(), 2)
    or_low  = round(or_bars['low'].min(), 2)
    or_mid  = round((or_high + or_low) / 2, 2)
    current_price = round(df['close'].iloc[-1], 2)

    # Target = 1× range, Stop = 0.3× range below/above entry
    range_size = or_high - or_low
    long_target  = round(or_high + range_size, 2)
    long_stop    = round(or_high - range_size * 0.3, 2)
    short_target = round(or_low  - range_size, 2)
    short_stop   = round(or_low  + range_size * 0.3, 2)

    if current_price > or_high:
        breakout = 'LONG_BREAKOUT'
        entry, target, stop = or_high, long_target, long_stop
    elif current_price < or_low:
        breakout = 'SHORT_BREAKOUT'
        entry, target, stop = or_low, short_target, short_stop
    else:
        breakout = 'INSIDE_RANGE'
        entry, target, stop = or_mid, long_target, long_stop

    rr = round(abs(target - entry) / abs(entry - stop), 2) if abs(entry - stop) > 0 else 0

    return {
        'ticker': ticker.upper(),
        'range_minutes': range_minutes,
        'or_high': or_high,
        'or_low': or_low,
        'or_mid': or_mid,
        'current_price': current_price,
        'breakout_status': breakout,
        'entry': entry,
        'target': target,
        'stop_loss': stop,
        'risk_reward': rr,
        'signal': 'BUY' if breakout == 'LONG_BREAKOUT' else 'SELL' if breakout == 'SHORT_BREAKOUT' else 'WAIT',
        'interpretation': (
            f'ORB({range_minutes}min): High={or_high}, Low={or_low}. '
            f'Current={current_price} — {breakout}. '
            f'Entry={entry}, Target={target}, Stop={stop}, R:R={rr}'
        )
    }


def compute_full_analysis(ticker: str) -> dict:
    """Run all indicators and return a consolidated summary."""
    vwap   = compute_vwap(ticker)
    rsi    = compute_rsi(ticker)
    macd   = compute_macd(ticker)
    bb     = compute_bollinger_bands(ticker)
    orb    = compute_opening_range(ticker)

    signals = [vwap['signal'], rsi['signal'], macd['signal'], bb['signal']]
    buy_count  = signals.count('BUY')
    sell_count = signals.count('SELL')

    consensus = (
        'STRONG_BUY'  if buy_count >= 3
        else 'BUY'    if buy_count == 2
        else 'STRONG_SELL' if sell_count >= 3
        else 'SELL'   if sell_count == 2
        else 'NEUTRAL'
    )

    return {
        'ticker': ticker.upper(),
        'consensus_signal': consensus,
        'indicator_signals': {
            'VWAP': vwap['signal'],
            'RSI': rsi['signal'],
            'MACD': macd['signal'],
            'Bollinger': bb['signal'],
            'ORB': orb['signal'],
        },
        'details': {'vwap': vwap, 'rsi': rsi, 'macd': macd, 'bollinger': bb, 'orb': orb}
    }


# Quick test
result = compute_full_analysis('RELIANCE')
print(f"Consensus signal for RELIANCE: {result['consensus_signal']}")
for ind, sig in result['indicator_signals'].items():
    print(f'  {ind:<12}: {sig}')

Consensus signal for RELIANCE: NEUTRAL
  VWAP        : NEUTRAL
  RSI         : HOLD
  MACD        : BUY
  Bollinger   : HOLD
  ORB         : WAIT


---
## 4. Strategy Signal Tools

Each strategy function returns a structured trade signal with entry, target, stop-loss, and rationale.

In [7]:
# ============================================================
# INTRADAY STRATEGY SIGNAL GENERATORS
# ============================================================

def vwap_reversion_signal(ticker: str, capital: float = 100000.0) -> dict:
    """
    VWAP Reversion Strategy:
    - BUY when price dips > 0.5% below VWAP (expect reversion to VWAP)
    - SELL when price rises > 0.5% above VWAP
    - Target: VWAP level; Stop: 0.3% beyond entry
    """
    vwap_data = compute_vwap(ticker)
    price = vwap_data['current_price']
    vwap  = vwap_data['vwap']
    dev   = vwap_data['deviation_pct']

    if dev < -0.5:
        direction = 'LONG'
        entry  = price
        target = vwap
        stop   = round(price * 0.997, 2)
        action = 'BUY'
    elif dev > 0.5:
        direction = 'SHORT'
        entry  = price
        target = vwap
        stop   = round(price * 1.003, 2)
        action = 'SELL_SHORT'
    else:
        return {
            'strategy': 'VWAP_REVERSION', 'ticker': ticker.upper(),
            'action': 'NO_TRADE', 'reason': f'Price within ±0.5% of VWAP (dev={dev:.3f}%)'
        }

    rr = round(abs(target - entry) / abs(entry - stop), 2) if abs(entry - stop) > 0 else 0
    qty = int(capital * 0.02 / abs(entry - stop)) if abs(entry - stop) > 0 else 0

    return {
        'strategy': 'VWAP_REVERSION',
        'ticker': ticker.upper(),
        'action': action,
        'direction': direction,
        'entry': entry,
        'target': target,
        'stop_loss': stop,
        'risk_reward': rr,
        'quantity': qty,
        'max_risk_rupees': round(abs(entry - stop) * qty, 2),
        'rationale': f'Price is {abs(dev):.3f}% {"below" if dev < 0 else "above"} VWAP. Mean-reversion to {vwap} expected.'
    }


def orb_signal(ticker: str, capital: float = 100000.0) -> dict:
    """
    Opening Range Breakout (ORB) Strategy:
    - Wait for first 30-minute range to form
    - BUY on break above OR High with target = 1× range
    - SELL on break below OR Low with target = 1× range
    """
    orb = compute_opening_range(ticker, range_minutes=30)
    signal = orb['signal']

    if signal == 'WAIT':
        return {
            'strategy': 'ORB', 'ticker': ticker.upper(),
            'action': 'WAIT',
            'reason': f'Price inside opening range ({orb["or_low"]}–{orb["or_high"]}). Wait for breakout.'
        }

    entry = orb['entry']
    stop  = orb['stop_loss']
    qty   = int(capital * 0.02 / abs(entry - stop)) if abs(entry - stop) > 0 else 0

    return {
        'strategy': 'ORB',
        'ticker': ticker.upper(),
        'action': signal,
        'direction': 'LONG' if signal == 'BUY' else 'SHORT',
        'entry': entry,
        'target': orb['target'],
        'stop_loss': stop,
        'risk_reward': orb['risk_reward'],
        'quantity': qty,
        'or_high': orb['or_high'],
        'or_low': orb['or_low'],
        'rationale': orb['interpretation']
    }


def rsi_macd_momentum_signal(ticker: str, capital: float = 100000.0) -> dict:
    """
    RSI + MACD Momentum Strategy:
    - BUY when RSI < 40 AND MACD is bullish
    - SELL when RSI > 60 AND MACD is bearish
    """
    rsi_data  = compute_rsi(ticker)
    macd_data = compute_macd(ticker)
    bb_data   = compute_bollinger_bands(ticker)

    rsi   = rsi_data['rsi']
    price = bb_data['current_price']
    macd_bull = 'BULLISH' in macd_data['crossover']
    macd_bear = 'BEARISH' in macd_data['crossover']

    if rsi < 40 and macd_bull:
        action = 'BUY'
        stop   = round(price * 0.995, 2)
        target = round(price * 1.012, 2)
    elif rsi > 60 and macd_bear:
        action = 'SELL_SHORT'
        stop   = round(price * 1.005, 2)
        target = round(price * 0.988, 2)
    else:
        return {
            'strategy': 'RSI_MACD_MOMENTUM', 'ticker': ticker.upper(),
            'action': 'NO_TRADE',
            'reason': f'No confluence: RSI={rsi:.1f}, MACD={macd_data["crossover"]}. Both must align.'
        }

    rr  = round(abs(target - price) / abs(price - stop), 2) if abs(price - stop) > 0 else 0
    qty = int(capital * 0.02 / abs(price - stop)) if abs(price - stop) > 0 else 0

    return {
        'strategy': 'RSI_MACD_MOMENTUM',
        'ticker': ticker.upper(),
        'action': action,
        'entry': price,
        'target': target,
        'stop_loss': stop,
        'risk_reward': rr,
        'quantity': qty,
        'rsi': rsi,
        'macd_crossover': macd_data['crossover'],
        'rationale': f'RSI={rsi:.1f} + MACD={macd_data["crossover"]} momentum confluence. Entry={price}, Target={target}, Stop={stop}'
    }


def bollinger_squeeze_signal(ticker: str, capital: float = 100000.0) -> dict:
    """
    Bollinger Band Squeeze Strategy:
    - Detect low volatility (bandwidth < 75% of 50-bar avg)
    - Trade the breakout direction when squeeze releases
    """
    bb = compute_bollinger_bands(ticker)
    vwap = compute_vwap(ticker)
    price = bb['current_price']

    if not bb['squeeze_detected']:
        return {
            'strategy': 'BB_SQUEEZE', 'ticker': ticker.upper(),
            'action': 'NO_TRADE',
            'reason': f'No squeeze. Bandwidth={bb["bandwidth_pct"]:.2f}%. Wait for compression.'
        }

    # Use VWAP to determine breakout direction bias
    if vwap['deviation_pct'] > 0:
        action = 'BUY'
        stop   = bb['lower_band']
        target = round(bb['upper_band'] + (bb['upper_band'] - bb['lower_band']), 2)
    else:
        action = 'SELL_SHORT'
        stop   = bb['upper_band']
        target = round(bb['lower_band'] - (bb['upper_band'] - bb['lower_band']), 2)

    rr  = round(abs(target - price) / abs(price - stop), 2) if abs(price - stop) > 0 else 0
    qty = int(capital * 0.015 / abs(price - stop)) if abs(price - stop) > 0 else 0

    return {
        'strategy': 'BB_SQUEEZE',
        'ticker': ticker.upper(),
        'action': action,
        'entry': price,
        'target': target,
        'stop_loss': round(stop, 2),
        'risk_reward': rr,
        'quantity': qty,
        'bandwidth_pct': bb['bandwidth_pct'],
        'squeeze_detected': True,
        'rationale': f'BB Squeeze active (BW={bb["bandwidth_pct"]:.2f}%). '
                     f'VWAP bias {vwap["position_vs_vwap"]} → {action} breakout play.'
    }


def scan_all_strategies(ticker: str, capital: float = 100000.0) -> dict:
    """Run all four strategies and return a consolidated trade recommendation."""
    signals = {
        'VWAP_REVERSION': vwap_reversion_signal(ticker, capital),
        'ORB': orb_signal(ticker, capital),
        'RSI_MACD': rsi_macd_momentum_signal(ticker, capital),
        'BB_SQUEEZE': bollinger_squeeze_signal(ticker, capital),
    }
    active_signals = {k: v for k, v in signals.items() if v.get('action') not in ('NO_TRADE', 'WAIT')}

    return {
        'ticker': ticker.upper(),
        'capital': capital,
        'active_strategies': len(active_signals),
        'signals': signals,
        'recommendation': (
            list(active_signals.values())[0] if len(active_signals) == 1
            else {'action': 'REVIEW_MULTIPLE', 'active': list(active_signals.keys())}
            if len(active_signals) > 1
            else {'action': 'NO_TRADE', 'reason': 'No strategy has a valid signal at this time.'}
        )
    }


# Quick test
test_scan = scan_all_strategies('TCS')
print(f"Active strategies for TCS: {test_scan['active_strategies']}")
for strat, sig in test_scan['signals'].items():
    print(f"  {strat:<20}: {sig.get('action', 'N/A')}")

Active strategies for TCS: 0
  VWAP_REVERSION      : NO_TRADE
  ORB                 : WAIT
  RSI_MACD            : NO_TRADE
  BB_SQUEEZE          : NO_TRADE


---
## 5. Risk Management Tools

In [8]:
# ============================================================
# RISK MANAGEMENT ENGINE
# Position sizing, risk-per-trade, portfolio-level controls
# ============================================================

def calculate_position_size(capital: float, entry: float, stop_loss: float,
                             risk_pct: float = 1.0, max_position_pct: float = 10.0) -> dict:
    """
    Kelly-inspired position sizing with hard caps.
    risk_pct: % of capital to risk per trade (default 1%)
    max_position_pct: max % of capital in single position (default 10%)
    """
    risk_amount = capital * (risk_pct / 100)
    per_share_risk = abs(entry - stop_loss)

    if per_share_risk <= 0:
        return {'error': 'Stop loss must differ from entry price'}

    raw_qty = risk_amount / per_share_risk
    max_qty = (capital * max_position_pct / 100) / entry
    final_qty = int(min(raw_qty, max_qty))
    position_value = round(final_qty * entry, 2)
    actual_risk = round(final_qty * per_share_risk, 2)

    return {
        'entry': entry,
        'stop_loss': stop_loss,
        'per_share_risk': round(per_share_risk, 2),
        'risk_amount_targeted': round(risk_amount, 2),
        'quantity': final_qty,
        'position_value': position_value,
        'actual_risk': actual_risk,
        'capital_deployed_pct': round(position_value / capital * 100, 2),
        'risk_pct_of_capital': round(actual_risk / capital * 100, 3),
        'status': 'APPROVED' if final_qty > 0 else 'REJECTED_ZERO_QTY'
    }


def evaluate_trade_risk(signal: dict, portfolio_state: dict = None) -> dict:
    """
    Evaluate a trade signal against risk rules.
    Returns APPROVED / REJECTED with reasons.
    """
    if portfolio_state is None:
        portfolio_state = {
            'capital': 100000.0,
            'daily_pnl': -500.0,
            'daily_pnl_limit': -2000.0,
            'open_positions': 2,
            'max_open_positions': 5,
            'trades_today': 3,
            'max_trades_today': 10
        }

    rejections = []
    warnings   = []

    # Rule 1: Daily loss limit
    if portfolio_state['daily_pnl'] <= portfolio_state['daily_pnl_limit']:
        rejections.append(f'DAILY_LOSS_LIMIT: P&L={portfolio_state["daily_pnl"]} <= limit={portfolio_state["daily_pnl_limit"]}')

    # Rule 2: Max open positions
    if portfolio_state['open_positions'] >= portfolio_state['max_open_positions']:
        rejections.append(f'MAX_POSITIONS: {portfolio_state["open_positions"]} positions open (max={portfolio_state["max_open_positions"]})')

    # Rule 3: Max trades per day
    if portfolio_state['trades_today'] >= portfolio_state['max_trades_today']:
        rejections.append(f'MAX_DAILY_TRADES: {portfolio_state["trades_today"]} trades today')

    # Rule 4: Minimum R:R ratio
    rr = signal.get('risk_reward', 0)
    if rr > 0 and rr < 1.5:
        warnings.append(f'LOW_RR: R:R={rr} (recommended >= 1.5)')

    # Rule 5: Position size check
    entry = signal.get('entry', 0)
    stop  = signal.get('stop_loss', 0)
    qty   = signal.get('quantity', 0)
    if entry > 0 and stop > 0:
        pos_val = entry * qty
        if pos_val > portfolio_state['capital'] * 0.20:
            warnings.append(f'LARGE_POSITION: Position value {pos_val:.0f} > 20% of capital')

    status = 'REJECTED' if rejections else 'APPROVED'

    return {
        'trade_signal': signal.get('strategy', 'UNKNOWN'),
        'ticker': signal.get('ticker', ''),
        'status': status,
        'rejections': rejections,
        'warnings': warnings,
        'portfolio_state': portfolio_state,
        'approved_quantity': signal.get('quantity', 0) if status == 'APPROVED' else 0,
        'summary': (
            f'Trade {status}. '
            + (f'Reasons: {"; ".join(rejections)}' if rejections else '')
            + (f' Warnings: {"; ".join(warnings)}' if warnings else '')
        )
    }


def compute_portfolio_exposure(positions: list) -> dict:
    """Compute current portfolio heat and sector/stock concentration."""
    if not positions:
        return {'total_positions': 0, 'total_exposure': 0, 'risk_level': 'LOW'}

    total_value = sum(p['value'] for p in positions)
    total_risk  = sum(p.get('risk', 0) for p in positions)
    capital     = 100000.0

    exposure_pct = round(total_value / capital * 100, 2)
    risk_pct     = round(total_risk  / capital * 100, 2)

    return {
        'total_positions': len(positions),
        'total_exposure': total_value,
        'exposure_pct_capital': exposure_pct,
        'total_risk': total_risk,
        'risk_pct_capital': risk_pct,
        'risk_level': 'HIGH' if risk_pct > 5 else 'MEDIUM' if risk_pct > 2.5 else 'LOW',
        'positions': positions
    }


print('Risk management tools loaded.')
# Demo
pos_size = calculate_position_size(100000, 2850.0, 2820.0)
print(f"Position size for RELIANCE: {pos_size['quantity']} shares, value={pos_size['position_value']}, risk={pos_size['actual_risk']}")

Risk management tools loaded.
Position size for RELIANCE: 3 shares, value=8550.0, risk=90.0


---
## 6. Performance Analytics Tools

In [9]:
# ============================================================
# PERFORMANCE ANALYTICS ENGINE
# Backtesting stats, P&L analysis, Sharpe, drawdown
# ============================================================

def simulate_strategy_backtest(strategy_name: str, ticker: str,
                                n_trades: int = 20, seed: int = 42) -> dict:
    """
    Simulate historical trades for a strategy and compute performance metrics.
    Uses realistic win-rates and R-multiples per strategy type.
    """
    np.random.seed(seed)

    # Strategy-specific parameters
    params = {
        'VWAP_REVERSION': {'win_rate': 0.58, 'avg_win_r': 1.2, 'avg_loss_r': 1.0},
        'ORB':            {'win_rate': 0.52, 'avg_win_r': 2.1, 'avg_loss_r': 1.0},
        'RSI_MACD':       {'win_rate': 0.55, 'avg_win_r': 1.8, 'avg_loss_r': 1.0},
        'BB_SQUEEZE':     {'win_rate': 0.50, 'avg_win_r': 2.5, 'avg_loss_r': 1.0},
    }
    p = params.get(strategy_name.upper(), params['ORB'])
    risk_per_trade = 500.0

    trades = []
    equity = 100000.0
    equity_curve = [equity]

    for i in range(n_trades):
        win = np.random.random() < p['win_rate']
        if win:
            pnl = risk_per_trade * p['avg_win_r'] * (0.7 + 0.6 * np.random.random())
        else:
            pnl = -risk_per_trade * p['avg_loss_r'] * (0.8 + 0.4 * np.random.random())
        equity += pnl
        equity_curve.append(equity)
        trades.append({'trade': i+1, 'result': 'WIN' if win else 'LOSS', 'pnl': round(pnl, 2)})

    wins = [t for t in trades if t['result'] == 'WIN']
    losses = [t for t in trades if t['result'] == 'LOSS']
    total_pnl = round(sum(t['pnl'] for t in trades), 2)
    win_rate_actual = round(len(wins) / n_trades * 100, 1)

    returns = np.diff(equity_curve) / equity_curve[:-1]
    sharpe = round(returns.mean() / (returns.std() + 1e-10) * np.sqrt(252), 2)

    peak = equity_curve[0]
    max_dd = 0.0
    for eq in equity_curve:
        if eq > peak:
            peak = eq
        dd = (peak - eq) / peak * 100
        if dd > max_dd:
            max_dd = dd

    profit_factor = (
        round(sum(t['pnl'] for t in wins) / abs(sum(t['pnl'] for t in losses)), 2)
        if losses and sum(t['pnl'] for t in losses) != 0 else 0
    )

    return {
        'strategy': strategy_name.upper(),
        'ticker': ticker.upper(),
        'n_trades': n_trades,
        'win_rate_pct': win_rate_actual,
        'total_pnl': total_pnl,
        'total_return_pct': round(total_pnl / 100000 * 100, 2),
        'avg_win': round(sum(t['pnl'] for t in wins) / max(len(wins), 1), 2),
        'avg_loss': round(sum(t['pnl'] for t in losses) / max(len(losses), 1), 2),
        'profit_factor': profit_factor,
        'sharpe_ratio': sharpe,
        'max_drawdown_pct': round(max_dd, 2),
        'final_equity': round(equity_curve[-1], 2),
        'equity_curve': [round(e, 2) for e in equity_curve],
        'recent_trades': trades[-5:]
    }


def compare_strategies(ticker: str) -> dict:
    """Backtest all four strategies on the same ticker and compare."""
    strategies = ['VWAP_REVERSION', 'ORB', 'RSI_MACD', 'BB_SQUEEZE']
    results = {}
    for s in strategies:
        results[s] = simulate_strategy_backtest(s, ticker)

    # Rank by Sharpe ratio
    ranked = sorted(results.items(), key=lambda x: x[1]['sharpe_ratio'], reverse=True)
    best = ranked[0]

    return {
        'ticker': ticker.upper(),
        'comparison': {
            s: {
                'win_rate': r['win_rate_pct'],
                'total_return': r['total_return_pct'],
                'sharpe': r['sharpe_ratio'],
                'max_dd': r['max_drawdown_pct'],
                'profit_factor': r['profit_factor']
            } for s, r in results.items()
        },
        'best_strategy': best[0],
        'best_sharpe': best[1]['sharpe_ratio'],
        'recommendation': f'{best[0]} is the best performing strategy for {ticker} with Sharpe={best[1]["sharpe_ratio"]}'
    }


def generate_performance_chart(ticker: str):
    """Generate and save a multi-strategy equity curve comparison chart."""
    strategies = ['VWAP_REVERSION', 'ORB', 'RSI_MACD', 'BB_SQUEEZE']
    colors = ['#2196F3', '#4CAF50', '#FF9800', '#E91E63']

    fig, axes = plt.subplots(2, 1, figsize=(12, 8))

    # Equity curves
    for strat, color in zip(strategies, colors):
        result = simulate_strategy_backtest(strat, ticker)
        axes[0].plot(result['equity_curve'], label=f'{strat} (Sharpe={result["sharpe_ratio"]})', color=color, linewidth=2)

    axes[0].axhline(y=100000, color='gray', linestyle='--', alpha=0.5, label='Starting Capital')
    axes[0].set_title(f'{ticker} — Strategy Equity Curves (20 Trades)', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Trade Number')
    axes[0].set_ylabel('Portfolio Value (₹)')
    axes[0].legend(fontsize=9)
    axes[0].grid(True, alpha=0.3)

    # Performance comparison bar chart
    comparison = compare_strategies(ticker)['comparison']
    metrics = ['win_rate', 'total_return', 'sharpe', 'profit_factor']
    x = np.arange(len(strategies))
    width = 0.2
    for i, metric in enumerate(metrics):
        vals = [comparison[s][metric] for s in strategies]
        axes[1].bar(x + i * width, vals, width, label=metric)

    axes[1].set_title('Strategy Performance Metrics Comparison', fontsize=13, fontweight='bold')
    axes[1].set_xticks(x + width * 1.5)
    axes[1].set_xticklabels([s.replace('_', ' ') for s in strategies], fontsize=9)
    axes[1].legend(fontsize=9)
    axes[1].grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    path = f'./trading_workspace/{ticker}_strategy_comparison.png'
    plt.savefig(path, dpi=120, bbox_inches='tight')
    plt.close()
    print(f'Chart saved: {path}')
    return path


# Quick test
comp = compare_strategies('HDFC')
print(f"Best strategy for HDFC: {comp['best_strategy']} (Sharpe={comp['best_sharpe']})")
for s, m in comp['comparison'].items():
    print(f"  {s:<20}: WR={m['win_rate']}%  Return={m['total_return']}%  Sharpe={m['sharpe']}")

Best strategy for HDFC: BB_SQUEEZE (Sharpe=10.53)
  VWAP_REVERSION      : WR=60.0%  Return=3.61%  Sharpe=5.32
  ORB                 : WR=60.0%  Return=9.19%  Sharpe=9.42
  RSI_MACD            : WR=60.0%  Return=7.33%  Sharpe=8.36
  BB_SQUEEZE          : WR=60.0%  Return=11.67%  Sharpe=10.53


---
## 7. Agent Definitions

Six specialized agents, each with domain expertise and tool access.

In [10]:
# ============================================================
# TRADING AGENT DEFINITIONS
# ============================================================

# ── Agent 1: Market Data Analyst ──────────────────────────────
market_data_agent = AssistantAgent(
    name='MarketDataAnalyst',
    system_message=(
        'You are an intraday market data specialist. Your responsibilities:\n'
        '1. Fetch intraday OHLCV data using generate_intraday_data()\n'
        '2. Provide market snapshots using get_market_snapshot()\n'
        '3. Report key price levels: open, high, low, current, volume\n'
        '4. Identify intraday price patterns and volume anomalies\n'
        '5. Always report data with timestamps\n\n'
        'Be concise. Format numbers clearly. Pass findings to TechnicalAnalyst.'
    ),
    model_client=model_client,
    tools=[generate_intraday_data, get_market_snapshot],
)

# ── Agent 2: Technical Analyst ─────────────────────────────────
technical_analyst = AssistantAgent(
    name='TechnicalAnalyst',
    system_message=(
        'You are a certified technical analyst specializing in intraday trading. Your responsibilities:\n'
        '1. Compute all technical indicators using compute_full_analysis()\n'
        '2. Analyze VWAP, RSI, MACD, Bollinger Bands, and ORB levels\n'
        '3. Identify chart patterns and indicator confluence\n'
        '4. Provide clear BUY/SELL/HOLD opinion with supporting indicators\n'
        '5. Always state which indicators agree and which conflict\n\n'
        'Use compute_vwap(), compute_rsi(), compute_macd(), compute_bollinger_bands(), '
        'compute_opening_range() for individual indicators or '
        'compute_full_analysis() for a complete picture.'
    ),
    model_client=model_client,
    tools=[compute_vwap, compute_rsi, compute_macd, compute_bollinger_bands,
           compute_opening_range, compute_full_analysis],
)

# ── Agent 3: Strategy Agent ────────────────────────────────────
strategy_agent = AssistantAgent(
    name='StrategyAgent',
    system_message=(
        'You are an intraday trading strategist. Your responsibilities:\n'
        '1. Evaluate four intraday strategies: VWAP Reversion, ORB, RSI+MACD, BB Squeeze\n'
        '2. Use scan_all_strategies() to get all signals at once\n'
        '3. Select the best strategy based on market conditions\n'
        '4. Provide specific entry, target, stop-loss levels\n'
        '5. Explain WHY a particular strategy fits current market conditions\n\n'
        'Available tools: vwap_reversion_signal(), orb_signal(), '
        'rsi_macd_momentum_signal(), bollinger_squeeze_signal(), scan_all_strategies()\n\n'
        'Always include: Entry price, Target, Stop-loss, Risk:Reward ratio, Quantity.'
    ),
    model_client=model_client,
    tools=[vwap_reversion_signal, orb_signal, rsi_macd_momentum_signal,
           bollinger_squeeze_signal, scan_all_strategies],
)

# ── Agent 4: Risk Manager ──────────────────────────────────────
risk_manager = AssistantAgent(
    name='RiskManager',
    system_message=(
        'You are the trading desk Risk Manager. Your responsibilities:\n'
        '1. Evaluate every trade signal against risk rules using evaluate_trade_risk()\n'
        '2. Compute appropriate position sizes using calculate_position_size()\n'
        '3. Check portfolio exposure using compute_portfolio_exposure()\n'
        '4. APPROVE or REJECT trades with clear reasoning\n'
        '5. Enforce: max 1% capital risk per trade, min 1.5 R:R ratio, max 5 open positions\n\n'
        'You have VETO POWER. If risk rules are violated, REJECT the trade even if the '
        'technical setup is good. Safety first.'
    ),
    model_client=model_client,
    tools=[calculate_position_size, evaluate_trade_risk, compute_portfolio_exposure],
)

# ── Agent 5: Trade Executor ────────────────────────────────────
trade_executor = AssistantAgent(
    name='TradeExecutor',
    system_message=(
        'You are the trade execution specialist. Your responsibilities:\n'
        '1. Accept APPROVED trade signals from RiskManager only\n'
        '2. Simulate order placement and provide execution confirmation\n'
        '3. Record trade with: timestamp, ticker, action, quantity, entry price, target, stop\n'
        '4. Monitor for exit triggers (target hit or stop hit)\n'
        '5. Report execution slippage if any\n\n'
        'Order format: ACTION QUANTITY TICKER @ PRICE | TARGET: X | STOP: X | R:R: X\n\n'
        'NEVER execute a trade that has not been approved by RiskManager.'
    ),
    model_client=model_client,
)

# ── Agent 6: Performance Analyst ──────────────────────────────
performance_analyst = AssistantAgent(
    name='PerformanceAnalyst',
    system_message=(
        'You are the trading performance analyst. Your responsibilities:\n'
        '1. Analyze strategy performance using simulate_strategy_backtest()\n'
        '2. Compare strategies using compare_strategies()\n'
        '3. Generate performance charts using generate_performance_chart()\n'
        '4. Compute: win rate, Sharpe ratio, max drawdown, profit factor\n'
        '5. Provide actionable recommendations to improve strategy performance\n\n'
        'Report format:\n'
        '- Performance Summary Table\n'
        '- Key Strengths of best strategy\n'
        '- Risk warnings\n'
        '- Improvement recommendations\n\n'
        'End your final report with ANALYSIS COMPLETE.'
    ),
    model_client=model_client,
    tools=[simulate_strategy_backtest, compare_strategies, generate_performance_chart],
)

print('All 6 trading agents initialized:')
for agent in [market_data_agent, technical_analyst, strategy_agent,
              risk_manager, trade_executor, performance_analyst]:
    print(f'  {agent.name}')

All 6 trading agents initialized:
  MarketDataAnalyst
  TechnicalAnalyst
  StrategyAgent
  RiskManager
  TradeExecutor
  PerformanceAnalyst


---
## 8. Implementation 1 — Two-Agent Technical Analysis Debate

**Scenario:** A `MarketDataAnalyst` fetches intraday data for a stock. A `TechnicalAnalyst` analyzes the data and debates the best trade setup. They iterate until the analyst reaches a confident recommendation.

In [ ]:
# ============================================================
# IMPLEMENTATION 1: Two-Agent Technical Analysis
# MarketDataAnalyst → TechnicalAnalyst debate
# ============================================================

team_impl1 = RoundRobinGroupChat(
    participants=[market_data_agent, technical_analyst],
    max_turns=6,
    termination_condition=TextMentionTermination('TERMINATE'),
)

task_impl1 = (
    'Analyze RELIANCE for an intraday trading opportunity right now.\n\n'
    'MarketDataAnalyst: Fetch current intraday data for RELIANCE and report:\n'
    '  - Current price, day high/low, volume trend\n'
    '  - First 5 bars of the session\n\n'
    'TechnicalAnalyst: Based on the data, compute all indicators and provide:\n'
    '  - VWAP analysis (price vs VWAP)\n'
    '  - RSI reading with overbought/oversold assessment\n'
    '  - MACD signal\n'
    '  - Bollinger Band position\n'
    '  - Overall BUY/SELL/HOLD recommendation\n\n'
    'Conclude with a clear, concise trading recommendation. End with TERMINATE.'
)

print('Running Two-Agent Technical Analysis for RELIANCE...')
print('=' * 60)
result_impl1 = await team_impl1.run(task=task_impl1)

print('\nConversation:')
for msg in result_impl1.messages:
    print(f'\n[{msg.source}]')
    print('-' * 40)
    content = msg.content if isinstance(msg.content, str) else str(msg.content)
    print(content[:1500] + '...' if len(content) > 1500 else content)

---
## 9. Implementation 2 — GroupChat Trading Floor (6 Agents)

**Scenario:** A full trading floor conversation — all 6 agents discuss a trade opportunity from market data to execution. The `SelectorGroupChat` routes messages to the most relevant agent based on context.

In [ ]:
# ============================================================
# IMPLEMENTATION 2: Full Trading Floor GroupChat
# 6 agents collaborate on a complete trade lifecycle
# ============================================================

# Use SelectorGroupChat for intelligent agent routing
trading_floor = SelectorGroupChat(
    participants=[
        market_data_agent,
        technical_analyst,
        strategy_agent,
        risk_manager,
        trade_executor,
        performance_analyst,
    ],
    model_client=model_client,
    termination_condition=TextMentionTermination('SESSION_CLOSED'),
    max_turns=14,
    selector_prompt=(
        'You are the Trading Floor Manager routing messages between specialists.\n'
        'Select the MOST APPROPRIATE agent based on the conversation context:\n'
        '- MarketDataAnalyst: when current price/volume data is needed\n'
        '- TechnicalAnalyst: when indicator analysis is needed\n'
        '- StrategyAgent: when trade signals and entry/exit levels are needed\n'
        '- RiskManager: when a trade signal needs approval\n'
        '- TradeExecutor: after RiskManager approval\n'
        '- PerformanceAnalyst: when reviewing strategy performance\n'
        'Never select the same agent twice consecutively unless critical.\n'
        'Available agents: {roles}\n'
        'Conversation history: {history}\n'
        'Select the next agent name from: {participants}'
    ),
)

task_impl2 = (
    'TRADING SESSION START — 10:30 AM\n'
    'We want to trade TCS today. Run the complete trade lifecycle:\n\n'
    '1. MarketDataAnalyst: Get TCS intraday data and market snapshot\n'
    '2. TechnicalAnalyst: Run full technical analysis on TCS\n'
    '3. StrategyAgent: Identify the best strategy and generate a trade signal\n'
    '4. RiskManager: Evaluate the signal (capital=100000, assume 2 open positions, daily P&L=-300)\n'
    '5. TradeExecutor: If approved, execute the trade with full order details\n'
    '6. PerformanceAnalyst: Give a quick backtest summary for the chosen strategy on TCS\n\n'
    'End the session with SESSION_CLOSED.'
)

print('Trading Floor GroupChat — TCS Trade Lifecycle')
print('=' * 60)
result_impl2 = await trading_floor.run(task=task_impl2)

print('\nTrading Floor Conversation:')
for msg in result_impl2.messages:
    print(f'\n[{msg.source}]')
    print('-' * 40)
    content = msg.content if isinstance(msg.content, str) else str(msg.content)
    print(content[:1200] + '...' if len(content) > 1200 else content)

---
## 10. Implementation 3 — Sequential Strategy Pipeline

**Scenario:** A sequential pipeline where each stage passes its output to the next. This pattern is ideal for systematic strategy evaluation across multiple tickers.

In [ ]:
# ============================================================
# IMPLEMENTATION 3: Sequential Multi-Ticker Strategy Pipeline
# Stage 1: Scan → Stage 2: Analyze best opportunity → Stage 3: Risk check
# ============================================================

# Stage 1: Market Scanner (data + strategy scan)
scanner_team = RoundRobinGroupChat(
    participants=[market_data_agent, strategy_agent],
    max_turns=4,
    termination_condition=TextMentionTermination('SCAN_COMPLETE'),
)

# Stage 2: Deep Analysis (technical + performance)
analysis_team = RoundRobinGroupChat(
    participants=[technical_analyst, performance_analyst],
    max_turns=4,
    termination_condition=TextMentionTermination('ANALYSIS_COMPLETE'),
)

# Stage 3: Risk + Execution
execution_team = RoundRobinGroupChat(
    participants=[risk_manager, trade_executor],
    max_turns=4,
    termination_condition=TextMentionTermination('TERMINATE'),
)

tickers_to_scan = ['HDFC', 'INFY', 'WIPRO']

print('=== STAGE 1: Market Scan ===' )
print('Scanning:', ', '.join(tickers_to_scan))
stage1_task = (
    f'Scan {tickers_to_scan} for intraday opportunities.\n'
    'MarketDataAnalyst: Get market snapshot for all three tickers.\n'
    'StrategyAgent: Run scan_all_strategies() on each ticker. '
    'Identify which ticker has the strongest signal.\n'
    'Conclude by naming the BEST_TICKER and BEST_STRATEGY. Say SCAN_COMPLETE.'
)
stage1_result = await scanner_team.run(task=stage1_task)
stage1_summary = stage1_result.messages[-1].content

print('\nStage 1 Result (final message):')
print(stage1_summary[:800])

print('\n=== STAGE 2: Deep Technical Analysis ===')
stage2_task = (
    f'Previous scan found the best opportunity. Context:\n{stage1_summary[:500]}\n\n'
    'TechnicalAnalyst: Run compute_full_analysis() on HDFC (use as the selected ticker).\n'
    'PerformanceAnalyst: Run compare_strategies() on HDFC and generate a chart.\n'
    'Provide a consensus view on trade quality. Say ANALYSIS_COMPLETE.'
)
stage2_result = await analysis_team.run(task=stage2_task)
stage2_summary = stage2_result.messages[-1].content

print('\nStage 2 Result (final message):')
print(stage2_summary[:800])

print('\n=== STAGE 3: Risk Assessment & Execution ===')
stage3_task = (
    f'Analysis complete. Context:\n{stage2_summary[:400]}\n\n'
    'RiskManager: Evaluate a hypothetical BUY trade on HDFC.\n'
    '  - Capital: 100,000\n'
    '  - Entry: current price from HDFC data (~1680)\n'
    '  - Stop: 1% below entry\n'
    '  - Target: 2% above entry\n'
    '  - Open positions: 1, Daily P&L: +200\n'
    'Use calculate_position_size() and evaluate_trade_risk().\n\n'
    'TradeExecutor: If approved, log the trade execution. Say TERMINATE.'
)
stage3_result = await execution_team.run(task=stage3_task)

print('\nStage 3 Result:')
for msg in stage3_result.messages:
    content = msg.content if isinstance(msg.content, str) else str(msg.content)
    print(f'\n[{msg.source}]: {content[:600]}')

---
## 11. Implementation 4 — Interactive Conversational Trading Assistant

**Scenario:** A natural-language trading assistant powered by all agents. Users ask conversational questions and the system routes to the right specialist. Demonstrates the full conversational nature of AutoGen.

Try questions like:
- *"What's the VWAP signal for ICICI right now?"*
- *"Should I use ORB or VWAP Reversion for WIPRO today?"*
- *"Compare all strategies for NVDA"*
- *"Run a full analysis on TSLA"*

In [ ]:
# ============================================================
# IMPLEMENTATION 4: Interactive Conversational Trading Assistant
# Full natural-language Q&A powered by all 6 agents
# ============================================================

async def ask_trading_assistant(query: str, verbose: bool = True) -> str:
    """
    Route a natural-language trading question to the appropriate agent team.
    Returns the final response as a string.
    """
    # Routing heuristics based on keywords
    query_lower = query.lower()

    if any(kw in query_lower for kw in ['backtest', 'compare', 'performance', 'sharpe', 'drawdown', 'win rate']):
        # Performance-focused query
        team = RoundRobinGroupChat(
            participants=[technical_analyst, performance_analyst],
            max_turns=4,
            termination_condition=TextMentionTermination('TERMINATE'),
        )
        system_prefix = 'Answer the following trading performance question. Use tools as needed. End with TERMINATE.\n\n'

    elif any(kw in query_lower for kw in ['risk', 'position size', 'stop', 'approve', 'reject']):
        # Risk-focused query
        team = RoundRobinGroupChat(
            participants=[strategy_agent, risk_manager],
            max_turns=4,
            termination_condition=TextMentionTermination('TERMINATE'),
        )
        system_prefix = 'Answer the following risk management question. Use tools as needed. End with TERMINATE.\n\n'

    elif any(kw in query_lower for kw in ['full analysis', 'complete analysis', 'full trade', 'trade setup']):
        # Full pipeline query
        team = SelectorGroupChat(
            participants=[market_data_agent, technical_analyst, strategy_agent, risk_manager],
            model_client=model_client,
            max_turns=8,
            termination_condition=TextMentionTermination('TERMINATE'),
        )
        system_prefix = 'Run a complete intraday trade analysis pipeline for the request below. End with TERMINATE.\n\n'

    else:
        # Default: technical analysis
        team = RoundRobinGroupChat(
            participants=[market_data_agent, technical_analyst, strategy_agent],
            max_turns=6,
            termination_condition=TextMentionTermination('TERMINATE'),
        )
        system_prefix = 'Answer the following intraday trading question. Use tools as needed. End with TERMINATE.\n\n'

    result = await team.run(task=system_prefix + query)

    if verbose:
        for msg in result.messages:
            print(f'\n[{msg.source}]')
            print('-' * 40)
            content = msg.content if isinstance(msg.content, str) else str(msg.content)
            print(content[:1000] + '...' if len(content) > 1000 else content)

    final = result.messages[-1].content if isinstance(result.messages[-1].content, str) else str(result.messages[-1].content)
    return final


# ============================================================
# DEMO CONVERSATION 1: VWAP Analysis
# ============================================================
print('QUERY 1: VWAP signal for ICICI')
print('=' * 60)
response1 = await ask_trading_assistant(
    'What is the VWAP signal for ICICI right now? '
    'Should I buy, sell, or hold based on VWAP reversion?'
)

In [ ]:
# ============================================================
# DEMO CONVERSATION 2: Strategy Comparison
# ============================================================
print('QUERY 2: Strategy comparison for WIPRO')
print('=' * 60)
response2 = await ask_trading_assistant(
    'Compare all four intraday strategies (VWAP Reversion, ORB, RSI+MACD, BB Squeeze) '
    'for WIPRO. Which has the best backtest performance? Generate a performance chart.'
)

In [ ]:
# ============================================================
# DEMO CONVERSATION 3: Full Trade Setup
# ============================================================
print('QUERY 3: Full trade setup for INFY')
print('=' * 60)
response3 = await ask_trading_assistant(
    'Run a full analysis and complete trade setup for INFY. '
    'I have a capital of 150000. Include: market data, technical indicators, '
    'best strategy signal, position sizing, and risk assessment.'
)

In [ ]:
# ============================================================
# DEMO CONVERSATION 4: Risk-focused query
# ============================================================
print('QUERY 4: Position sizing with risk management')
print('=' * 60)
response4 = await ask_trading_assistant(
    'I want to trade the ORB strategy on HDFC. My capital is 200000 and I have '
    '3 open positions already with a daily P&L of -800. '
    'Calculate the position size (risking 1% capital) and tell me if the risk manager approves.'
)

---
## 12. Visualization — Intraday Chart with Indicators

In [ ]:
# ============================================================
# INTRADAY CHART WITH ALL INDICATORS
# Standalone visualization (no agent needed)
# ============================================================

def plot_intraday_analysis(ticker: str, figsize=(14, 10)):
    """Plot intraday OHLCV with VWAP, Bollinger Bands, RSI, and MACD."""
    df = _bars_to_df(ticker)

    # Compute indicators
    typical_price = (df['high'] + df['low'] + df['close']) / 3
    df['vwap'] = (typical_price * df['volume']).cumsum() / df['volume'].cumsum()

    df['bb_mid'] = df['close'].rolling(20).mean()
    df['bb_std'] = df['close'].rolling(20).std()
    df['bb_upper'] = df['bb_mid'] + 2 * df['bb_std']
    df['bb_lower'] = df['bb_mid'] - 2 * df['bb_std']

    delta = df['close'].diff()
    gain = delta.clip(lower=0).ewm(alpha=1/14, min_periods=14).mean()
    loss = (-delta.clip(upper=0)).ewm(alpha=1/14, min_periods=14).mean()
    df['rsi'] = 100 - (100 / (1 + gain / loss.replace(0, 1e-10)))

    ema12 = df['close'].ewm(span=12, adjust=False).mean()
    ema26 = df['close'].ewm(span=26, adjust=False).mean()
    df['macd'] = ema12 - ema26
    df['macd_signal'] = df['macd'].ewm(span=9, adjust=False).mean()
    df['macd_hist'] = df['macd'] - df['macd_signal']

    x = range(len(df))
    xticks_step = max(1, len(df) // 12)
    xtick_labels = [df['timestamp'].iloc[i] if i % xticks_step == 0 else '' for i in x]

    fig, axes = plt.subplots(3, 1, figsize=figsize, gridspec_kw={'height_ratios': [3, 1, 1]})
    fig.suptitle(f'{ticker} — Intraday Analysis (5-min bars)', fontsize=14, fontweight='bold')

    # ── Panel 1: Price + VWAP + Bollinger Bands ──
    ax1 = axes[0]
    ax1.plot(x, df['close'], color='#1976D2', linewidth=1.5, label='Close')
    ax1.plot(x, df['vwap'],  color='#FF6F00', linewidth=1.5, linestyle='--', label='VWAP')
    ax1.plot(x, df['bb_upper'], color='#9C27B0', linewidth=1, linestyle=':', label='BB Upper')
    ax1.plot(x, df['bb_mid'],   color='#9C27B0', linewidth=0.8, linestyle='-', alpha=0.5)
    ax1.plot(x, df['bb_lower'], color='#9C27B0', linewidth=1, linestyle=':', label='BB Lower')
    ax1.fill_between(x, df['bb_upper'], df['bb_lower'], alpha=0.05, color='#9C27B0')

    # Volume bars on secondary axis
    ax1v = ax1.twinx()
    ax1v.bar(x, df['volume'], alpha=0.15, color='#78909C', width=0.8)
    ax1v.set_ylabel('Volume', fontsize=8, color='#78909C')
    ax1v.tick_params(axis='y', labelsize=7, labelcolor='#78909C')

    ax1.set_ylabel('Price', fontsize=10)
    ax1.legend(loc='upper left', fontsize=8)
    ax1.grid(True, alpha=0.3)
    ax1.set_xticks(list(x)[::xticks_step])
    ax1.set_xticklabels([df['timestamp'].iloc[i] for i in x if i % xticks_step == 0], fontsize=7, rotation=45)

    # ── Panel 2: RSI ──
    ax2 = axes[1]
    ax2.plot(x, df['rsi'], color='#E91E63', linewidth=1.5, label='RSI(14)')
    ax2.axhline(70, color='red',  linewidth=0.8, linestyle='--', alpha=0.7)
    ax2.axhline(30, color='green', linewidth=0.8, linestyle='--', alpha=0.7)
    ax2.axhline(50, color='gray',  linewidth=0.5, linestyle='-', alpha=0.3)
    ax2.fill_between(x, df['rsi'], 70, where=(df['rsi'] >= 70), alpha=0.2, color='red', label='Overbought')
    ax2.fill_between(x, df['rsi'], 30, where=(df['rsi'] <= 30), alpha=0.2, color='green', label='Oversold')
    ax2.set_ylabel('RSI', fontsize=10)
    ax2.set_ylim(0, 100)
    ax2.legend(loc='upper left', fontsize=8)
    ax2.grid(True, alpha=0.3)
    ax2.set_xticks(list(x)[::xticks_step])
    ax2.set_xticklabels([df['timestamp'].iloc[i] for i in x if i % xticks_step == 0], fontsize=7, rotation=45)

    # ── Panel 3: MACD ──
    ax3 = axes[2]
    ax3.plot(x, df['macd'],        color='#1976D2', linewidth=1.5, label='MACD')
    ax3.plot(x, df['macd_signal'], color='#FF9800', linewidth=1.2, label='Signal')
    colors_hist = ['#4CAF50' if h >= 0 else '#F44336' for h in df['macd_hist']]
    ax3.bar(x, df['macd_hist'], color=colors_hist, alpha=0.7, width=0.8, label='Histogram')
    ax3.axhline(0, color='gray', linewidth=0.5)
    ax3.set_ylabel('MACD', fontsize=10)
    ax3.legend(loc='upper left', fontsize=8)
    ax3.grid(True, alpha=0.3)
    ax3.set_xticks(list(x)[::xticks_step])
    ax3.set_xticklabels([df['timestamp'].iloc[i] for i in x if i % xticks_step == 0], fontsize=7, rotation=45)

    plt.tight_layout()
    path = f'./trading_workspace/{ticker}_intraday_chart.png'
    plt.savefig(path, dpi=120, bbox_inches='tight')
    plt.close()
    print(f'Chart saved: {path}')
    return path


# Generate charts for all key tickers
for ticker in ['RELIANCE', 'TCS', 'HDFC']:
    plot_intraday_analysis(ticker)
print('\nAll intraday charts generated in ./trading_workspace/')

---
## 13. Best Practices for Trading Multi-Agent Systems

### Agent Design

| Principle | Why It Matters |
|---|---|
| **One agent, one responsibility** | Market data, analysis, strategy, risk, execution — each has a clear domain |
| **Risk agent has VETO power** | Trading without risk controls leads to blown accounts |
| **Tool outputs must be structured** | Agents downstream depend on predictable data formats (dicts, not free text) |
| **Always include termination phrases** | Without `TERMINATE`, conversations run indefinitely and cost tokens |
| **Use SelectorGroupChat for complex routing** | RoundRobin doesn't scale when not every agent needs to speak every turn |

### Trading-Specific Considerations

| Rule | Explanation |
|---|---|
| **Never paper-trade untested strategies** | Backtest first; live trading requires proven edge |
| **Risk per trade ≤ 1% of capital** | Protects against drawdowns; 20 consecutive losses still leaves 80% capital |
| **Minimum 1.5:1 Risk:Reward** | Ensures profitability even with 40% win rate |
| **Respect daily loss limits** | If you lose 2% in a day, stop trading — emotional decisions follow losses |
| **Intraday positions must close by 3:20 PM** | Avoid overnight gap risk on intraday strategies |

### Common Pitfalls

| Pitfall | Cause | Fix |
|---|---|---|
| Agents generate conflicting signals | No consensus mechanism | Use `compute_full_analysis()` for multi-indicator consensus |
| Trade executed without risk check | Missing RiskManager in pipeline | Always route through RiskManager before TradeExecutor |
| Backtest overfitting | Parameters tuned on same data | Walk-forward test; use out-of-sample data |
| Agents loop endlessly | No termination condition | Set both `TextMentionTermination` and `MaxMessageTermination` |
| High API costs | Too many agents in GroupChat | Use Sequential Chat for defined pipelines; GroupChat for open-ended discussion |

### Architecture Decision Guide

```
Use Case                           → Recommended Pattern
─────────────────────────────────────────────────────────
Single ticker quick analysis       → Two-Agent Chat (Data + Technical)
Complete trade lifecycle           → SelectorGroupChat (all 6 agents)
Multi-ticker systematic scan       → Sequential Chat (Scan → Analyze → Execute)
Natural language Q&A               → Routed Single-Agent (keyword dispatch)
Strategy optimization debate       → RoundRobin (Technical + Strategy + Performance)
```

---
## Summary

| Component | Description |
|---|---|
| **6 Specialized Agents** | MarketData, TechnicalAnalyst, StrategyAgent, RiskManager, TradeExecutor, PerformanceAnalyst |
| **4 Intraday Strategies** | VWAP Reversion, Opening Range Breakout, RSI+MACD Momentum, Bollinger Band Squeeze |
| **5 Technical Indicators** | VWAP, RSI(14), MACD(12,26,9), Bollinger Bands(20,2), ORB(30) |
| **Risk Controls** | Position sizing, daily loss limits, R:R validation, portfolio exposure monitoring |
| **4 Conversation Patterns** | Two-Agent, GroupChat, Sequential Pipeline, Conversational Q&A |
| **Backtesting** | Simulated performance metrics: win rate, Sharpe, max drawdown, profit factor |